In [ ]:
!pip install pandas numpy datasets transformers tqdm scikit-learn matplotlib scipy

In [ ]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset

# =====================================================================
# 1. CONFIGURATION GLOBALE ET ARBORESCENCE DES DOSSIERS
# =====================================================================
DOSSIER_MERE = "INLP_Experiments"

# Paramètres pour le cas d'étude "Black and White"
suffix = "black_white"
nom_categorie = "Identité (Black vs White)"

# Dossiers
DOSSIER_ATTRIBUT = f"{DOSSIER_MERE}/{suffix}"
os.makedirs(DOSSIER_ATTRIBUT, exist_ok=True)

print(f"Architecture prête :")
print(f" 📂 {DOSSIER_MERE}/")
print(f" ┗━ 📂 {suffix}/  <- Matrices, graphiques et vecteurs")

In [ ]:
# =====================================================================
# 2. CHARGEMENT ET PREPROCESSING (JIGSAW VIA HUGGING FACE)
# =====================================================================
cols_to_keep = ['comment_text'] + ['black', 'white', 'toxicity']
df_jigsaw = pd.read_csv('/home/onyxia/work/Stat_App/Data/JIGSAW/all_data.csv', usecols=cols_to_keep)

# Filtrer pour ne garder que les lignes où les attributs de race sont annotés
df_jigsaw = df_jigsaw.dropna(subset=['black', 'white', 'toxicity'])

# On définit des variables binaires pour identifier les mentions claires
df_jigsaw['is_black'] = (df_jigsaw['black'] >= 0.5).astype(int)
df_jigsaw['is_white'] = (df_jigsaw['white'] >= 0.5).astype(int)

# On filtre les commentaires pour garder ceux qui mentionnent explicitement
# l'un ou l'autre de manière exclusive (OU exclusif)
df_filtered = df_jigsaw[(df_jigsaw['is_black'] == 1) ^ (df_jigsaw['is_white'] == 1)].copy()

# Définition de Y (Tâche : Toxicité)
df_filtered['y_toxicity'] = (df_filtered['toxicity'] >= 0.5).astype(int)

# Définition de Z (Biais : Identité Black ou White)
# 1 pour Black, 0 pour White
df_filtered['z_race'] = df_filtered['is_black'] 

# Équilibrage : On crée un sous-échantillon avec un ratio 1:1 pour éviter 
# que le modèle ne se concentre que sur la classe majoritaire.
df_black = df_filtered[df_filtered['z_race'] == 1]
df_white = df_filtered[df_filtered['z_race'] == 0]

min_len = min(len(df_black), len(df_white))
df_final = pd.concat([
    df_black.sample(min_len, random_state=42),
    df_white.sample(min_len, random_state=42)
]).sample(frac=1, random_state=42)


texts_jigsaw = df_final['comment_text'].tolist()
y_jigsaw = df_final['y_toxicity'].values
z_jigsaw = df_final['z_race'].values

print(f"\nNombre total de textes conservés : {len(df_final)}")
print(f" - Toxiques (Y=1) : {y_jigsaw.sum()}")
print(f" - Mentions 'Black' (Z=1) : {z_jigsaw.sum()}")
print(f" - Mentions 'White' (Z=0) : {len(z_jigsaw) - z_jigsaw.sum()}")